# Passo 3: Vídeo em Tempo Real e Inteligência Artificial (MediaPipe)

Você tocou num ponto excelente! Até agora trabalhamos com 1 frame (uma foto). 
Mas **o que é um vídeo?** É simplesmente um laço de repetição (`loop`) que tira dezenas de fotos por segundo e analisa uma a uma.

Neste notebook vamos:
1. Criar um loop para processar as imagens da câmera como vídeo.
2. Usar o **MediaPipe** (uma biblioteca fantástica de IA criada pelo Google) para detectar esqueletos humanos e mapear as juntas das mãos.

In [4]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
from IPython.display import clear_output

# IMPORTANTE: Cole sua URL com usuário e senha novamente
RTSP_URL = 'rtsp://admin:@Jeff2712@10.64.0.60:554/'

# --- SETUP DA INTELIGÊNCIA ARTIFICIAL (MediaPipe) ---
# Inicializa os módulos de desenho e os modelos de Mão (Hands) e Corpo (Pose)
mp_desenho = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

### O Loop de Vídeo
Como estamos dentro do Jupyter Notebook (no navegador), se fizermos um loop infinito o seu computador vai travar. 
Então, vamos configurar o loop para capturar e processar apenas os próximos **60 frames** da sua câmera (alguns segundos de vídeo).

**O que fazer:** Rode a célula abaixo, coloque a mão na frente da câmera da bancada, faça um legalzinho (👍) e veja a mágica!

In [5]:
cap = cv2.VideoCapture(RTSP_URL)

try:
    # Vamos analisar apenas 60 frames (fotos) em sequência para simular o vídeo
    for i in range(60):
        ret, frame = cap.read()
        if not ret:
            print("Falha ao capturar vídeo da câmera.")
            break
            
        # A IA do MediaPipe precisa que a imagem esteja em RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # --- A MÁGICA DA IA ACONTECE AQUI ---
        resultado_maos = hands.process(frame_rgb)
        resultado_corpo = pose.process(frame_rgb)
        
        # Se a IA achar mãos, nós desenhamos os 21 pontos (landmarks) de articulação
        if resultado_maos.multi_hand_landmarks:
            for mao in resultado_maos.multi_hand_landmarks:
                mp_desenho.draw_landmarks(frame_rgb, mao, mp_hands.HAND_CONNECTIONS)
                
        # Se a IA achar um corpo humano, nós desenhamos o esqueleto
        if resultado_corpo.pose_landmarks:
            mp_desenho.draw_landmarks(frame_rgb, resultado_corpo.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # --- MOSTRANDO NO NAVEGADOR ---
        plt.figure(figsize=(8,6))
        plt.imshow(frame_rgb)
        plt.title(f'Frame processado: {i+1}/60')
        plt.axis('off')
        
        # O "truque" do Jupyter para vídeo: mostra a imagem atual e imediatamente limpa para a próxima
        plt.show()
        clear_output(wait=True)
        
finally:
    # Ao final do loop (ou se você interromper), liberamos a câmera e a memória da IA
    cap.release()
    hands.close()
    pose.close()
    print("Fim da captura e processamento do vídeo!")

Fim da captura e processamento do vídeo!
